# 03 · Input pipeline check (plan step 2.5)

Checks the `tf.data` pipeline (`plantdx.data.pipeline`) on the real images with the settings of
`configs/base.yaml`:

1. one batch from each split: shapes, dtype, value range, one-hot labels;
2. augmentation examples on real leaves → `reports/figures/aug_examples.jpg` (look at them!);
3. speed: training batches per second, and the val set before and after caching;
4. memory used by the cached val/test sets.

Results are written to `reports/pipeline_check.json`. Select a **T4 GPU** runtime (the speed numbers
are meant for the training setup). About 10 minutes.

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
!git log -1 --oneline
!nvidia-smi -L

## 2 · Images

In [ ]:
from plantdx.data.images import download_archive, extract_variant

VARIANT, DATA_ROOT = "color", "/content/data"
print("new files extracted:", extract_variant(download_archive(), DATA_ROOT, VARIANT))

## 3 · Datasets from the config and one batch per split

In [ ]:
import json
import numpy as np
import tensorflow as tf

from plantdx.config import load_config, set_seed
from plantdx.data.pipeline import datasets_from_config, load_split

cfg = load_config("configs/base.yaml")
set_seed(cfg["seed"])
ds = datasets_from_config(cfg, DATA_ROOT)
batch_size = cfg["train"]["batch_size"]

check = {"tensorflow": tf.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")],
         "batch_size": batch_size, "image_size": cfg["data"]["image_size"], "splits": {}}
for split in ("train", "val", "test"):
    n = len(load_split(split))
    x, y = next(iter(ds[split]))
    info = {
        "images": n,
        "batches_per_epoch": int(np.ceil(n / batch_size)),
        "x_shape": list(x.shape), "x_dtype": x.dtype.name,
        "x_min": float(tf.reduce_min(x)), "x_max": float(tf.reduce_max(x)),
        "y_shape": list(y.shape), "y_rows_sum_to_1": bool(np.allclose(y.numpy().sum(1), 1)),
        "classes_in_first_batch": int(len(set(y.numpy().argmax(1)))),
    }
    check["splits"][split] = info
    print(split, info)

assert check["splits"]["train"]["x_shape"] == [batch_size, 224, 224, 3]
assert all(s["x_min"] >= 0 and s["x_max"] <= 255 and s["y_rows_sum_to_1"] for s in check["splits"].values())
print("\nPASS: shapes, value range and labels")

## 4 · Augmentation examples ← **look at this figure**

Each row: the image as the model sees it at evaluation, then 5 training versions. Check that the leaf and its
lesions stay recognisable (no lesion cut away, no unnatural colours) and that rotated corners are filled
plausibly.

In [ ]:
from pathlib import Path
from IPython.display import Image as IPImage, display

from plantdx.data.pipeline import AugmentConfig
from plantdx.viz import plot_augmentations

FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
out = plot_augmentations(load_split("train"), DATA_ROOT, VARIANT, FIG / "aug_examples.jpg",
                         augment=AugmentConfig.from_dict(cfg["augment"]), n_images=6, n_aug=5, seed=cfg["seed"])
display(IPImage(str(out)))

## 5 · Speed and memory

In [ ]:
import os
import time
import psutil

def images_per_second(dataset, n_batches, warmup=5):
    it = iter(dataset)
    for _ in range(warmup):
        next(it)
    t0, n = time.perf_counter(), 0
    for _ in range(n_batches):
        x, _ = next(it)
        n += int(x.shape[0])
    return n / (time.perf_counter() - t0)

train_ips = images_per_second(ds["train"], 100)
check["train_images_per_second"] = round(train_ips, 1)
check["train_epoch_seconds_input_only"] = round(check["splits"]["train"]["images"] / train_ips, 1)

rss0 = psutil.Process().memory_info().rss
t0 = time.perf_counter(); n_val = sum(int(x.shape[0]) for x, _ in ds["val"]); first = time.perf_counter() - t0
t0 = time.perf_counter(); sum(int(x.shape[0]) for x, _ in ds["val"]); second = time.perf_counter() - t0
rss1 = psutil.Process().memory_info().rss
check["val_pass_seconds"] = {"first_fills_cache": round(first, 1), "second_from_cache": round(second, 1)}
check["val_cache_ram_gb"] = round((rss1 - rss0) / 1e9, 2)
check["ram_total_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
check["cpu_count"] = os.cpu_count()

print(json.dumps({k: v for k, v in check.items() if k != "splits"}, indent=2))
Path("reports/pipeline_check.json").write_text(json.dumps(check, indent=2) + "\n")

**How to read it.** `train_epoch_seconds_input_only` is how long one epoch would take if the model cost
nothing; if it is much longer than a training epoch later, the input pipeline is the bottleneck (then set
`data.cache_train` to a local file such as `/content/cache/train`). The val pass from cache should be much faster
than the first one.

## 6 · Download the results

In [ ]:
from google.colab import files

!rm -f /content/step2_pipeline.zip && zip -q /content/step2_pipeline.zip reports/pipeline_check.json reports/figures/aug_examples.jpg
files.download("/content/step2_pipeline.zip")